In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as prep_mammo
from tensorflow.keras.applications.resnet50 import preprocess_input as prep_us
import warnings
warnings.filterwarnings('ignore')

print('TF Version:', tf.__version__)
print('GPUs Available:', len(tf.config.list_physical_devices('GPU')))

PAIRS_CSV_PATH = '/kaggle/input/datasets/habibashhefny/pairsmammo-ultrasound/all_pairs (1).csv' 

MAMMO_MODEL_PATH = '/kaggle/input/notebooks/habibashhefny/fine-tuning-efficient-mammoram-on-cbis-ddsm/mammo_ft_B_last30.keras' 

US_MODEL_PATH = '/kaggle/input/notebooks/habibashhefny/ultrasound-classification-fine-tuned/output/best_phase2.keras' 

print("Setup and Imports Done!")

In [ ]:
print("Loading Mammogram model...")
try:
 mammo_model = load_model(MAMMO_MODEL_PATH, compile=False)
 print("Mammogram model loaded.")
except Exception as e:
 print(f" Error loading Mammogram model: {e}")

print("\nLoading Ultrasound model...")
try:
 us_model = load_model(US_MODEL_PATH, compile=False)
 print("Ultrasound model loaded.")
except Exception as e:
 print(f" Error loading Ultrasound model: {e}")

print("\n Model Info:")
print(f" Mammo Input Shape: {mammo_model.input_shape}")
print(f" US Input Shape: {us_model.input_shape}")

In [ ]:
def prepare_mammo_image(img_path):
 img = cv2.imread(img_path)
 if img is None:
 raise ValueError(f"Could not read Mammo image: {img_path}")
 
 img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
 
 img = cv2.resize(img, (456, 456), interpolation=cv2.INTER_CUBIC)
 
 img_array = np.array(img, dtype=np.float32)
 return prep_mammo(img_array)

def prepare_us_image(img_path):
 try:
 img = Image.open(img_path).convert('RGB')
 except Exception as e:
 raise ValueError(f"Could not read US image: {img_path} | Error: {e}")
 
 img = img.resize((224, 224))
 
 img_array = np.array(img, dtype=np.float32)
 return prep_us(img_array)

print("Preprocessing functions defined successfully!")

In [ ]:
import os
import cv2

MAMMO_BASE_DIR = '/kaggle/input/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset'
sample_path_from_csv = "jpeg/1.3.6.1.4.1.9590.100.1.2.332220304112291686205901600122715192543/1-270.jpg"

full_path = os.path.join(MAMMO_BASE_DIR, sample_path_from_csv)
print(f"Testing path: {full_path}")

if os.path.exists(full_path):
 print("Success! Image found.")
 img = cv2.imread(full_path)
 if img is not None:
 print(f"Image loaded correctly. Shape: {img.shape}")
else:
 print("Still not found. Check if the CSV path starts with 'CBIS-DDSM/' and remove it from the join.")

In [ ]:
import os
from tqdm.auto import tqdm
import numpy as np

pairs_df = pd.read_csv(PAIRS_CSV_PATH)
print(f"Total pairs to process: {len(pairs_df)}")

MAMMO_BASE_DIR = '/kaggle/input/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset'

ps_mammo_img_list = []
ps_us_img_list = []

print("\nStarting Live Inference... (This might take a while)")

for index, row in tqdm(pairs_df.iterrows(), total=len(pairs_df)):
 try:
 raw_path = row['mammo_path']
 clean_path = raw_path.replace('CBIS-DDSM/', '') 
 mammo_path = os.path.join(MAMMO_BASE_DIR, clean_path)
 
 m_img = prepare_mammo_image(mammo_path)
 m_img_batch = np.expand_dims(m_img, axis=0) 
 
 m_preds = mammo_model.predict(m_img_batch, verbose=0)
 
 if m_preds.shape[-1] > 1:
 m_ps = float(m_preds[0][1]) 
 else:
 m_ps = float(m_preds[0][0]) 
 
 # ── 2. US Inference ──
 us_path = row['us_path'] 
 u_img = prepare_us_image(us_path)
 u_img_batch = np.expand_dims(u_img, axis=0)
 
 u_preds = us_model.predict(u_img_batch, verbose=0)
 
 if u_preds.shape[-1] > 1:
 u_ps = float(u_preds[0][1])
 else:
 u_ps = float(u_preds[0][0])
 
 ps_mammo_img_list.append(m_ps)
 ps_us_img_list.append(u_ps)
 
 except Exception as e:
 print(f"Error processing row {index}: {e}")
 ps_mammo_img_list.append(np.nan)
 ps_us_img_list.append(np.nan)

pairs_df['PS_mammo_img'] = ps_mammo_img_list
pairs_df['PS_us_img'] = ps_us_img_list

initial_len = len(pairs_df)
pairs_df = pairs_df.dropna(subset=['PS_mammo_img', 'PS_us_img']).reset_index(drop=True)

print(f"\nInference completed! Successfully processed {len(pairs_df)} / {initial_len} pairs.")

INFERENCE_OUTPUT_PATH = '/kaggle/working/synthetic_pairs_final_live.csv'
pairs_df.to_csv(INFERENCE_OUTPUT_PATH, index=False)
print(f" Saved live inference results to: {INFERENCE_OUTPUT_PATH}")

display(pairs_df[['mammo_path', 'PS_mammo', 'PS_mammo_img', 'us_path', 'PS_us', 'PS_us_img', 'label', 'split']].head())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

train_df = pairs_df[pairs_df['split'] == 'ft_train']
val_df = pairs_df[pairs_df['split'] == 'ft_val']
test_df = pairs_df[pairs_df['split'] == 'final_test']

features = ['PS_mammo_img', 'PS_us_img']

X_train, y_train = train_df[features], train_df['label']
X_val, y_val = val_df[features], val_df['label']
X_test, y_test = test_df[features], test_df['label']

print(f"Mode: IMAGE-ONLY FUSION with CROSS-VALIDATION")
print(f" Samples: Train={len(X_train)} | Val={len(X_val)} | Test={len(X_test)}")

# cross validation 
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}
base_model = LogisticRegression()

grid_search = GridSearchCV(base_model, param_grid, cv=5, scoring='roc_auc')
grid_search.fit(X_train, y_train)

fusion_model = grid_search.best_estimator_

print("\n" + "*"*40)
print(f"Best Hyperparameter found by CV: C = {grid_search.best_params_['C']}")
print("*"*40)

# Evaluation
y_prob_val = fusion_model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, y_prob_val)

y_prob_test = fusion_model.predict_proba(X_test)[:, 1]
y_pred_test = fusion_model.predict(X_test)
test_auc = roc_auc_score(y_test, y_prob_test)

print("\n" + "="*40)
print(f"Validation AUC (Images Only): {val_auc:.4f}")
print(f"FINAL TEST AUC (Images Only): {test_auc:.4f}")
print("="*40)

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr_mammo, tpr_mammo, _ = roc_curve(y_test, test_df['PS_mammo_img'])
fpr_us, tpr_us, _ = roc_curve(y_test, test_df['PS_us_img'])
fpr_fusion, tpr_fusion, _ = roc_curve(y_test, y_prob_test)

plt.figure(figsize=(8, 6))
plt.plot(fpr_mammo, tpr_mammo, label=f'Mammogram Only (AUC = {auc(fpr_mammo, tpr_mammo):.2f})')
plt.plot(fpr_us, tpr_us, label=f'Ultrasound Only (AUC = {auc(fpr_us, tpr_us):.2f})')
plt.plot(fpr_fusion, tpr_fusion, label=f'Image-Only Fusion (AUC = {test_auc:.2f})', linewidth=3, linestyle='--')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Comparison: Individual Models vs. Image Fusion')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

fpr, tpr, thresholds = roc_curve(y_val, y_prob_val)

j_scores = tpr - fpr
best_idx = np.argmax(j_scores)

print(f" Best Threshold (from Val): {best_threshold:.4f}")
print(f" Sensitivity (TPR): {tpr[best_idx]:.4f}")
print(f" Specificity (1-FPR): {1 - fpr[best_idx]:.4f}")

y_pred_test_tuned = (y_prob_test >= best_threshold).astype(int)

plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_test_tuned)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu',
 xticklabels=['Benign', 'Malignant'],
 yticklabels=['Benign', 'Malignant'])
plt.title(f'Tuned Threshold = {best_threshold:.3f}\nAUC: {test_auc:.4f}')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_test_tuned,
 target_names=['Benign', 'Malignant']))

# comparison of three models 


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

models_comparison = {
 'Mammogram Only': test_df['PS_mammo_img'],
 'Ultrasound Only': test_df['PS_us_img'],
 'Late Fusion (Ours)': y_prob_test,
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Model Comparison — Individual vs Fusion (Optimized Thresholds)', fontsize=16, fontweight='bold')

results = {}

for ax, (model_name, probs) in zip(axes, models_comparison.items()):
 
 if model_name == 'Mammogram Only':
 val_probs = val_df['PS_mammo_img']
 elif model_name == 'Ultrasound Only':
 val_probs = val_df['PS_us_img']
 else:
 val_probs = y_prob_val

 fpr_, tpr_, thresh_ = roc_curve(y_val, val_probs)
 j_scores = tpr_ - fpr_ 
 best_idx = np.argmax(j_scores)
 best_thresh = thresh_[best_idx] 

 auc_score = roc_auc_score(y_test, probs)
 y_pred_ = (probs >= best_thresh).astype(int) 
 cm_ = confusion_matrix(y_test, y_pred_)
 
 tn, fp, fn, tp = cm_.ravel()
 sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
 specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

 results[model_name] = {
 'AUC': auc_score,
 'Sensitivity': sensitivity,
 'Specificity': specificity,
 'Threshold': best_thresh
 }

 sns.heatmap(cm_, annot=True, fmt='d', cmap='YlGnBu' if model_name != 'Late Fusion (Ours)' else 'RdPu', ax=ax,
 xticklabels=['Benign', 'Malignant'],
 yticklabels=['Benign', 'Malignant'])
 
 ax.set_title(f'{model_name}\nAUC={auc_score:.4f} | Sens={sensitivity:.2%} | Spec={specificity:.2%}', fontsize=11)
 ax.set_ylabel('Actual')
 ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

print("\n" + "="*55)
print(f"{'Model':<25} {'AUC':>7} {'Sensitivity':>12} {'Specificity':>12}")
print("="*55)
for name, m in results.items():
 marker = " ⭐" if name == 'Late Fusion (Ours)' else ""
 print(f"{name:<25} {m['AUC']:>7.4f} {m['Sensitivity']:>11.2%} {m['Specificity']:>11.2%}{marker}")
print("="*55)